# Qwen Image 2.1 private TPU training session

This notebook is generated and submitted by the local session orchestrator for the full 40,000-update eight-replica run. Attach the private source, paired-cache, model-assets and `nerfviking/qwen21-fizgig-private-auth` inputs. The dedicated private auth dataset supplies `kaggle_token`; no Kaggle Secret is required. A separate smoke run is optional.


In [ ]:
import time
allocation_started_unix = time.time()
AITK_SESSION_CONFIG = {}


In [ ]:
import hashlib, importlib.metadata as metadata, json, os, pathlib, platform, subprocess, sys, uuid, zipfile
root = pathlib.Path('/kaggle/working') / ('qwen21_source_' + uuid.uuid4().hex)
root.mkdir()
inputs = pathlib.Path('/kaggle/input')
archives = list(inputs.rglob('kaggle_tpu_full_bundle.zip'))
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)): raise RuntimeError('Duplicate source ZIP members')
        manifest = json.loads(archive.read('source_manifest.json'))
        if set(names) != set(manifest['files']) | {'source_manifest.json'}: raise RuntimeError('Undeclared source ZIP members')
        for name, digest in manifest['files'].items():
            path = pathlib.PurePosixPath(name)
            if path.is_absolute() or '..' in path.parts or chr(92) in name: raise RuntimeError('Unsafe source path')
            content = archive.read(name)
            if hashlib.sha256(content).hexdigest() != digest: raise RuntimeError('Source hash mismatch: ' + name)
            destination = root / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(content)
elif not archives:
    candidates = [p for p in inputs.rglob('source_manifest.json') if json.loads(p.read_text()).get('kind') == 'qwen21_full_model_source']
    if len(candidates) != 1: raise RuntimeError('Attach exactly one full-model source bundle')
    manifest = json.loads(candidates[0].read_text())
    for name, digest in manifest['files'].items():
        path = pathlib.PurePosixPath(name)
        if path.is_absolute() or '..' in path.parts or chr(92) in name: raise RuntimeError('Unsafe source path')
        content = (candidates[0].parent / name).read_bytes()
        if hashlib.sha256(content).hexdigest() != digest: raise RuntimeError('Source hash mismatch: ' + name)
        destination = root / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
else: raise RuntimeError('Attach exactly one full-model source bundle')
if manifest.get('schema') != 1 or manifest.get('kind') != 'qwen21_full_model_source': raise RuntimeError('Wrong source manifest')
(root / 'source_manifest.json').write_text(json.dumps(manifest, indent=2))
versions = {name: metadata.version(name) for name in ('torch', 'torch-xla', 'libtpu')}
print('Runtime:', sys.version, platform.platform(), versions)
if any(versions[name].split('+')[0] != expected for name, expected in (('torch', '2.8.0'), ('torch-xla', '2.8.0'), ('libtpu', '0.0.17'))):
    raise RuntimeError('Select the previously verified TPU image: Torch/XLA 2.8.0 + libtpu 0.0.17; this notebook never replaces runtime wheels')
if sys.version_info[:2] not in ((3, 10), (3, 11), (3, 12)): raise RuntimeError('Pinned repository dependencies require Python 3.10-3.12')
sys.path.insert(0, str(root))
from workers.kaggle_tpu.runtime_env import configure_torch_only
configure_torch_only()
env = os.environ.copy()
for name in ('TPU_PROCESS_ADDRESSES', 'CLOUD_TPU_TASK_ID'): env.pop(name, None)
env['PJRT_DEVICE'] = 'TPU'
env['NO_ALBUMENTATIONS_UPDATE'] = '1'
preflight = subprocess.run([sys.executable, '-c', 'import torch_xla.runtime as xr; assert xr.device_type() == "TPU"; n=xr.addressable_device_count(); assert n == 8, n; print("TPU logical devices:", n)'], env=env, text=True, capture_output=True)
print(preflight.stdout)
if preflight.returncode: raise RuntimeError('TPU preflight failed: ' + preflight.stderr)
print('Verified source commit:', manifest['source_commit'], 'files:', len(manifest['files']))


In [ ]:
constraints = root / 'runtime-constraints.txt'
constraints.write_text('\n'.join(name + '==' + value for name, value in versions.items()) + '\n')
install_log = pathlib.Path('/kaggle/working/qwen21_dependency_install.log')
with install_log.open('w') as log:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--no-deps', '--index-url', 'https://download.pytorch.org/whl/cpu', 'torchvision==0.23.0+cpu', 'torchaudio==2.8.0+cpu'], stdout=log, stderr=subprocess.STDOUT, check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--constraint', str(constraints), '--requirement', str(root / 'workers/kaggle_tpu/requirements_smoke.txt')], stdout=log, stderr=subprocess.STDOUT, check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=log, stderr=subprocess.STDOUT, check=True)
after = {name: metadata.version(name) for name in versions}
if after != versions: raise RuntimeError('Dependency setup altered a protected runtime wheel')
print('Dependencies installed; log:', install_log)
import_log = pathlib.Path('/kaggle/working/qwen21_import_preflight.log')
with import_log.open('w') as log:
    subprocess.run([sys.executable, '-m', 'workers.kaggle_tpu.runtime_env'], cwd=root,
                   env=dict(env, PJRT_DEVICE='CPU'), stdout=log, stderr=subprocess.STDOUT, check=True)
print('CPU training imports passed; log:', import_log)
cache_archives = list(inputs.rglob('paired_cache_bundle.zip'))
if len(cache_archives) > 1: raise RuntimeError('Attach exactly one paired-cache archive')
if cache_archives:
    sys.path.insert(0, str(root))
    from workers.kaggle_tpu.cache_bundle import extract_cache_bundle
    extracted_cache = extract_cache_bundle(cache_archives[0], pathlib.Path('/kaggle/working') / ('qwen21_cache_' + uuid.uuid4().hex))
    cache_manifest = extracted_cache / 'manifest.json'
cache_manifests = []
for path in inputs.rglob('manifest.json'):
    try:
        if json.loads(path.read_text()).get('kind') == 'qwen21_fizgig_paired_cache': cache_manifests.append(path)
    except (ValueError, UnicodeError): continue
asset_manifests = list(inputs.rglob('asset-manifest.json'))
if (not cache_archives and len(cache_manifests) != 1) or len(asset_manifests) != 1: raise RuntimeError('Attach exactly one paired cache and one model asset dataset')
if not cache_archives: cache_manifest = cache_manifests[0]
asset_manifest = asset_manifests[0]
print('Cache:', cache_manifest.parent, 'Assets:', asset_manifest.parent)


In [ ]:
sys.path.insert(0, str(root))
token_paths = [path for path in inputs.rglob('kaggle_token')
               if path.parent.name == 'qwen21-fizgig-private-auth' and path.is_file()]
if len(token_paths) != 1:
    raise RuntimeError('Attach exactly one private qwen21-fizgig-private-auth dataset containing kaggle_token')
try:
    secret_token = token_paths[0].read_bytes().decode('ascii')
except UnicodeDecodeError:
    raise RuntimeError('Private auth token must contain ASCII text') from None
if secret_token.endswith('\r\n'):
    secret_token = secret_token[:-2]
elif secret_token.endswith('\n'):
    secret_token = secret_token[:-1]
if not secret_token or any(character.isspace() for character in secret_token):
    raise RuntimeError('Private auth token must be nonempty with no whitespace except one terminal newline')
os.environ['KAGGLE_API_TOKEN'] = secret_token
env['KAGGLE_API_TOKEN'] = secret_token
del secret_token
config = AITK_SESSION_CONFIG
if not config['session_id'] or not config['job_id'] or not config['attempt_id']:
    raise RuntimeError('The local orchestrator must inject a session config')
if config['resume_version'] is None:
    if config['resume_step'] != 0 or config['resume_manifest_sha256'] is not None:
        raise RuntimeError('First session has an inconsistent resume marker')
    resume = None
else:
    from workers.kaggle_tpu.checkpoint_relay import KaggleDataset, install_download
    relay_input = pathlib.Path('/kaggle/working') / ('relay_input_' + config['session_id'])
    relay_input.mkdir()
    download = relay_input / 'download'
    download.mkdir()
    KaggleDataset(config['relay_dataset']).download(config['resume_version'], download)
    installed = install_download(download, relay_input, job_id=config['job_id'],
                                 dataset_ref=config['relay_dataset'], version=config['resume_version'])
    if (installed['step'] != config['resume_step'] or
            installed['manifest_sha256'] != config['resume_manifest_sha256'] or
            installed['dataset_version'] != config['resume_version']):
        raise RuntimeError('Pinned resume checkpoint differs from local verified relay record')
    resume = installed['checkpoint_path']
smoke_reports = []
if config.get('smoke_report_sha256'):
    for path in inputs.rglob('diagnostic.json'):
        try:
            item = json.loads(path.read_text())
        except (ValueError, UnicodeError):
            continue
        if (hashlib.sha256(path.read_bytes()).hexdigest() == config['smoke_report_sha256'] and
                item.get('status') == 'pass' and item.get('world_size') == 1 and
                item.get('checkpoint', {}).get('adapter_restored')):
            smoke_reports.append(path)
    if len(smoke_reports) != 1:
        raise RuntimeError('Attach exactly one passing one-rank full-model diagnostic')
output = pathlib.Path('/kaggle/working') / ('training_' + config['attempt_id'])
command = [sys.executable, str(root / 'workers/kaggle_tpu/train_replicated.py'),
           '--cache', str(cache_manifest.parent), '--asset-manifest', str(asset_manifest),
           '--job-id', config['job_id'],
           '--attempt-id', config['attempt_id'], '--relay-dataset', config['relay_dataset'],
           '--output', str(output), '--allocation-started-unix', str(allocation_started_unix)]
if smoke_reports:
    command.extend(['--smoke-report', str(smoke_reports[0])])
if resume:
    command.extend(['--resume', str(resume)])
else:
    command.append('--relay-create')
print('Starting private eight-replica session', config['session_id'], 'from step', config['resume_step'])
with pathlib.Path('/kaggle/working/training-supervisor.log').open('w') as log:
    result = subprocess.run(command, cwd=root, env=env, stdout=log, stderr=subprocess.STDOUT)
if result.returncode:
    raise RuntimeError('Training supervisor failed; inspect private training-supervisor.log and result.json')
print('Training supervisor completed; result:', output / 'result.json')
